1

In [1]:
!pip install -q datasets scikit-learn pandas numpy matplotlib seaborn tensorflow

2

In [2]:
from datasets import load_dataset
import pandas as pd
import numpy as np

ds = load_dataset("yaful/MAGE")

print(ds)
print("\nTraining columns:", ds["train"].column_names)
print("\nTraining example:")
print(ds["train"][0])

README.md:   0%|          | 0.00/1.93k [00:00<?, ?B/s]

train.csv: reconstructing file:   0%|          |  0.00B /  404MB            

train.csv: downloading bytes:           |  0.00B            

valid.csv: reconstructing file:   0%|          |  0.00B / 72.3MB            

valid.csv: downloading bytes:           |  0.00B            

test.csv: reconstructing file:   0%|          |  0.00B / 71.7MB            

... (3 outputs left out)

Generating train split:   0%|          | 0/319071 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/56792 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/60743 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 319071
    })
    validation: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 56792
    })
    test: Dataset({
        features: ['text', 'label', 'src'],
        num_rows: 60743
    })
})

Training columns: ['text', 'label', 'src']

Training example:
{'text': "White girls very rarely date Asian men. Even in Asia white men are much more popular than Asian men. Even though an Asian guy may have a preference for white girls he doesn't have much of a chance with one so he should instead pursue a non-White girl. Asian guys are penalized by their looks because I don't see them as being significantly less social, athletic, intelligent, wealthy, etc. Facial looks are genetic and cannot be changed easily. Having a fit body helps but not nearly as much as for white or black guys. I wish it weren't true. EDIT: Basically I'm saying Asian men are at a significant disadvantage in

3

In [3]:
import pandas as pd
import numpy as np

# Convert splits to DataFrames for inspection
train_df = ds["train"].to_pandas()
val_df = ds["validation"].to_pandas()
test_df = ds["test"].to_pandas()

# 1. Split sizes and class distributions
for split_name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n--- {split_name} ---")
    print("Rows:", len(df))
    print("\nLabel counts:")
    print(df["label"].value_counts().sort_index())
    print("\nLabel proportions:")
    print(df["label"].value_counts(normalize=True).sort_index().round(4))

# 2. Source distribution in training data
print("\n--- Training sources ---")
print(train_df["src"].value_counts())

# 3. Missing values
print("\n--- Missing values ---")
for split_name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{split_name}:")
    print(df.isna().sum())

# 4. Empty or whitespace-only texts
for split_name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    empty_count = df["text"].fillna("").astype(str).str.strip().eq("").sum()
    print(f"{split_name} empty texts: {empty_count}")

# 5. Exact duplicate text counts within each split
for split_name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    duplicate_count = df["text"].duplicated().sum()
    print(f"{split_name} duplicate texts: {duplicate_count}")


--- Train ---
Rows: 319071

Label counts:
label
0    225753
1     93318
Name: count, dtype: int64

Label proportions:
label
0    0.7075
1    0.2925
Name: proportion, dtype: float64

--- Validation ---
Rows: 56792

Label counts:
label
0    27993
1    28799
Name: count, dtype: int64

Label proportions:
label
0    0.4929
1    0.5071
Name: proportion, dtype: float64

--- Test ---
Rows: 60743

Label counts:
label
0    30478
1    30265
Name: count, dtype: int64

Label proportions:
label
0    0.5018
1    0.4982
Name: proportion, dtype: float64

--- Training sources ---
src
yelp_human                               31827
eli5_human                               16706
squad_human                              15820
wp_human                                  6356
xsum_human                                4708
                                         ...  
tldr_machine_continuation_gpt_j            473
squad_machine_continuation_gpt_neox        455
sci_gen_machine_continuation_gpt_j         433
yel

4

In [4]:
# 1. Confirm label mapping from source names
print("Training label/source combinations:")
print(
    train_df.groupby(["label", "src"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

# 2. Check whether duplicate test texts have conflicting labels
test_duplicates = test_df[test_df["text"].duplicated(keep=False)].copy()

print("\nDuplicate test rows:", len(test_duplicates))
print("Unique duplicated test texts:", test_duplicates["text"].nunique())

print("\nLabels among duplicated test texts:")
print(
    test_duplicates.groupby("text")["label"]
    .nunique()
    .value_counts()
    .rename(index={1: "Same label", 2: "Conflicting labels"})
)

# 3. Check exact text overlap across splits
train_texts = set(train_df["text"])
val_texts = set(val_df["text"])
test_texts = set(test_df["text"])

print("\nTrain-validation overlap:", len(train_texts & val_texts))
print("Train-test overlap:", len(train_texts & test_texts))
print("Validation-test overlap:", len(val_texts & test_texts))

Training label/source combinations:
label  src                                    
1      yelp_human                                 31827
       eli5_human                                 16706
       squad_human                                15820
       wp_human                                    6356
       xsum_human                                  4708
       sci_gen_human                               4436
       cmv_human                                   4223
       roct_human                                  3287
       hswag_human                                 3129
       tldr_human                                  2826
0      xsum_machine_continuation_t0_3b              800
       xsum_machine_topical_gpt-3.5-trubo           800
       xsum_machine_continuation_gpt-3.5-trubo      800
       roct_machine_specified_gpt-3.5-trubo         800
       wp_machine_topical_text-davinci-002          800
       xsum_machine_specified_gpt-3.5-trubo         800
       roct_machine_s

5

In [5]:
import pandas as pd
import numpy as np

# -----------------------------------------------
# 1. Remove missing and blank texts
# -----------------------------------------------

def basic_clean(df):
    df = df.copy()
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str).str.strip()
    df = df[df["text"] != ""]
    return df

train_clean = basic_clean(train_df)
val_clean = basic_clean(val_df)
test_clean = basic_clean(test_df)

# -----------------------------------------------
# 2. Remove exact duplicates within each split
# -----------------------------------------------

train_clean = train_clean.drop_duplicates(subset=["text"]).reset_index(drop=True)
val_clean = val_clean.drop_duplicates(subset=["text"]).reset_index(drop=True)
test_clean = test_clean.drop_duplicates(subset=["text"]).reset_index(drop=True)

# -----------------------------------------------
# 3. Remove validation texts that appear in test
# -----------------------------------------------

test_text_set = set(test_clean["text"])

val_before = len(val_clean)
val_clean = val_clean[~val_clean["text"].isin(test_text_set)].reset_index(drop=True)

print("Validation rows removed due to test overlap:", val_before - len(val_clean))

# -----------------------------------------------
# 4. Confirm no exact overlap remains
# -----------------------------------------------

train_text_set = set(train_clean["text"])
val_text_set = set(val_clean["text"])

print("Train-validation overlap:", len(train_text_set & val_text_set))
print("Train-test overlap:", len(train_text_set & set(test_clean["text"])))
print("Validation-test overlap:", len(val_text_set & set(test_clean["text"])))

# -----------------------------------------------
# 5. Sample exactly 5,050 examples per class
# -----------------------------------------------

human_df = train_clean[train_clean["label"] == 1]
ai_df = train_clean[train_clean["label"] == 0]

n_human = len(human_df)

ai_sample = ai_df.sample(
    n=n_human,
    random_state=42
)

balanced_train = (
    pd.concat([human_df, ai_sample], ignore_index=True)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print("\nBalanced training size:", len(balanced_train))
print("\nBalanced label counts:")
print(balanced_train["label"].value_counts().sort_index())

print("\nFinal split sizes:")
print("Train:", len(balanced_train))
print("Validation:", len(val_clean))
print("Test:", len(test_clean))

Validation rows removed due to test overlap: 52
Train-validation overlap: 0
Train-test overlap: 0
Validation-test overlap: 0

Balanced training size: 186636

Balanced label counts:
label
0    93318
1    93318
Name: count, dtype: int64

Final split sizes:
Train: 186636
Validation: 56740
Test: 59981


6

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text = balanced_train["text"].tolist()
y_train = balanced_train["label"].to_numpy()

X_val_text = val_clean["text"].tolist()
y_val = val_clean["label"].to_numpy()

X_test_text = test_clean["text"].tolist()
y_test = test_clean["label"].to_numpy()

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=100000,
    min_df=2,
    sublinear_tf=True,
    dtype=np.float32
)

X_train_tfidf = tfidf.fit_transform(X_train_text)
X_val_tfidf = tfidf.transform(X_val_text)
X_test_tfidf = tfidf.transform(X_test_text)

print("Train matrix:", X_train_tfidf.shape)
print("Validation matrix:", X_val_tfidf.shape)
print("Test matrix:", X_test_tfidf.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

Train matrix: (186636, 100000)
Validation matrix: (56740, 100000)
Test matrix: (59981, 100000)
Vocabulary size: 100000


7

In [7]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver="liblinear",
    random_state=42
)

logreg.fit(X_train_tfidf, y_train)

print("Logistic Regression training complete.")

Logistic Regression training complete.


8

In [8]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

val_preds = logreg.predict(X_val_tfidf)
val_probs = logreg.predict_proba(X_val_tfidf)

print("Validation accuracy:", accuracy_score(y_val, val_preds))
print("\nClassification report:")
print(classification_report(
    y_val,
    val_preds,
    target_names=["AI-generated", "Human-written"],
    digits=4
))
print("Validation ROC-AUC:", roc_auc_score(y_val, val_probs[:, 1]))

Validation accuracy: 0.7328516038068382

Classification report:
               precision    recall  f1-score   support

 AI-generated     0.7260    0.7365    0.7312     27991
Human-written     0.7397    0.7293    0.7345     28749

     accuracy                         0.7329     56740
    macro avg     0.7329    0.7329    0.7328     56740
 weighted avg     0.7329    0.7329    0.7329     56740

Validation ROC-AUC: 0.8175388800198706


9

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

results = []

for C in [0.1, 0.5, 1.0, 2.0, 5.0]:
    model = LogisticRegression(
        C=C,
        max_iter=1000,
        solver="liblinear",
        random_state=42
    )

    model.fit(X_train_tfidf, y_train)

    preds = model.predict(X_val_tfidf)
    probs = model.predict_proba(X_val_tfidf)[:, 1]

    results.append({
        "C": C,
        "Accuracy": accuracy_score(y_val, preds),
        "Macro_F1": f1_score(y_val, preds, average="macro"),
        "ROC_AUC": roc_auc_score(y_val, probs)
    })

results_df = pd.DataFrame(results)
print(results_df.sort_values("ROC_AUC", ascending=False).to_string(index=False))

  C  Accuracy  Macro_F1  ROC_AUC
5.0  0.749912  0.749142 0.833843
2.0  0.743638  0.743414 0.827715
1.0  0.732852  0.732841 0.817539
0.5  0.716655  0.716568 0.801605
0.1  0.665333  0.663522 0.744162


10

In [10]:
for C in [7.5, 10.0]:
    model = LogisticRegression(
        C=C,
        max_iter=1000,
        solver="liblinear",
        random_state=42
    )

    model.fit(X_train_tfidf, y_train)

    preds = model.predict(X_val_tfidf)
    probs = model.predict_proba(X_val_tfidf)[:, 1]

    print(
        f"C={C:<5} "
        f"Accuracy={accuracy_score(y_val, preds):.4f} "
        f"Macro-F1={f1_score(y_val, preds, average='macro'):.4f} "
        f"ROC-AUC={roc_auc_score(y_val, probs):.4f}"
    )

C=7.5   Accuracy=0.7505 Macro-F1=0.7495 ROC-AUC=0.8345
C=10.0  Accuracy=0.7506 Macro-F1=0.7494 ROC-AUC=0.8344


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import numpy as np

# Character-level TF-IDF
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    max_features=100000,
    min_df=3,
    sublinear_tf=True,
    dtype=np.float32
)

X_train_char = char_vectorizer.fit_transform(X_train_text)
X_val_char = char_vectorizer.transform(X_val_text)
X_test_char = char_vectorizer.transform(X_test_text)

print("Character TF-IDF shapes:")
print("Train:", X_train_char.shape)
print("Validation:", X_val_char.shape)
print("Test:", X_test_char.shape)

# Combine existing word TF-IDF with character TF-IDF
X_train_combined = hstack(
    [X_train_tfidf, X_train_char],
    format="csr"
)

X_val_combined = hstack(
    [X_val_tfidf, X_val_char],
    format="csr"
)

X_test_combined = hstack(
    [X_test_tfidf, X_test_char],
    format="csr"
)

print("Combined feature shape:", X_train_combined.shape)

Character TF-IDF shapes:
Train: (186636, 100000)
Validation: (56740, 100000)
Test: (59981, 100000)
Combined feature shape: (186636, 200000)


In [12]:
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score
)

svm_results = []

for C in [0.1, 1.0, 2.0]:

    svm_model = LinearSVC(
        C=C,
        max_iter=5000,
        random_state=42
    )

    svm_model.fit(X_train_combined, y_train)

    val_preds = svm_model.predict(X_val_combined)
    val_scores = svm_model.decision_function(X_val_combined)

    svm_results.append({
        "C": C,
        "Accuracy": accuracy_score(y_val, val_preds),
        "Macro_F1": f1_score(y_val, val_preds, average="macro"),
        "ROC_AUC": roc_auc_score(y_val, val_scores)
    })

svm_results_df = pd.DataFrame(svm_results)

print(
    svm_results_df
    .sort_values("ROC_AUC", ascending=False)
    .to_string(index=False)
)

best_svm_C = svm_results_df.sort_values(
    "ROC_AUC", ascending=False
).iloc[0]["C"]

print("\nBest SVM C:", best_svm_C)

  C  Accuracy  Macro_F1  ROC_AUC
1.0  0.775467  0.774245 0.859644
2.0  0.772453  0.770976 0.856679
0.1  0.762443  0.762368 0.849271

Best SVM C: 1.0


In [13]:
from sklearn.metrics import classification_report, confusion_matrix

svm_final = LinearSVC(
    C=float(best_svm_C),
    max_iter=5000,
    random_state=42
)

svm_final.fit(X_train_combined, y_train)

svm_test_preds = svm_final.predict(X_test_combined)
svm_test_scores = svm_final.decision_function(X_test_combined)

print("SVM Test Accuracy:",
      accuracy_score(y_test, svm_test_preds))

print("SVM Test ROC-AUC:",
      roc_auc_score(y_test, svm_test_scores))

print("\nClassification Report:")
print(classification_report(
    y_test,
    svm_test_preds,
    target_names=["AI-generated", "Human-written"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, svm_test_preds))

SVM Test Accuracy: 0.7545556092762709
SVM Test ROC-AUC: 0.8413880370795024

Classification Report:
               precision    recall  f1-score   support

 AI-generated     0.8087    0.6772    0.7371     30478
Human-written     0.7145    0.8345    0.7698     29503

     accuracy                         0.7546     59981
    macro avg     0.7616    0.7558    0.7535     59981
 weighted avg     0.7623    0.7546    0.7532     59981

Confusion Matrix:
[[20640  9838]
 [ 4884 24619]]


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, classification_report, confusion_matrix
)

lr_final = LogisticRegression(
    C=7.5,
    max_iter=1000,
    solver="liblinear",
    random_state=42
)

lr_final.fit(X_train_tfidf, y_train)

lr_test_probs = lr_final.predict_proba(X_test_tfidf)[:, 1]
lr_test_preds = (lr_test_probs >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, lr_test_preds))
print("ROC-AUC:", roc_auc_score(y_test, lr_test_probs))
print(classification_report(
    y_test, lr_test_preds,
    target_names=["AI-generated", "Human-written"]
))
print("Confusion matrix:\n", confusion_matrix(y_test, lr_test_preds))

Accuracy: 0.736983378069722
ROC-AUC: 0.8216744687378008
               precision    recall  f1-score   support

 AI-generated       0.78      0.68      0.72     30478
Human-written       0.71      0.80      0.75     29503

     accuracy                           0.74     59981
    macro avg       0.74      0.74      0.74     59981
 weighted avg       0.74      0.74      0.74     59981

Confusion matrix:
 [[20617  9861]
 [ 5915 23588]]


In [15]:
!pip install -q gensim

import gensim.downloader as api

print("Downloading pretrained GloVe embeddings...")

glove_vectors = api.load("glove-wiki-gigaword-100")

print("GloVe loaded successfully.")
print("Embedding dimension:", glove_vectors.vector_size)

[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe loaded successfully.
Embedding dimension: 100


In [16]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CNN_VOCAB_SIZE = 30000
CNN_MAX_LEN = 256
EMBEDDING_DIM = 100

cnn_tokenizer = Tokenizer(
    num_words=CNN_VOCAB_SIZE,
    oov_token="<OOV>"
)

cnn_tokenizer.fit_on_texts(X_train_text)

X_train_seq = cnn_tokenizer.texts_to_sequences(X_train_text)
X_val_seq = cnn_tokenizer.texts_to_sequences(X_val_text)
X_test_seq = cnn_tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=CNN_MAX_LEN,
    padding="post",
    truncating="post"
)

X_val_pad = pad_sequences(
    X_val_seq,
    maxlen=CNN_MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=CNN_MAX_LEN,
    padding="post",
    truncating="post"
)

# Construct pretrained embedding matrix
word_index = cnn_tokenizer.word_index

embedding_matrix = np.zeros(
    (CNN_VOCAB_SIZE, EMBEDDING_DIM),
    dtype=np.float32
)

found_words = 0

for word, index in word_index.items():

    if index >= CNN_VOCAB_SIZE:
        continue

    if word in glove_vectors:
        embedding_matrix[index] = glove_vectors[word]
        found_words += 1

print("Training shape:", X_train_pad.shape)
print("Validation shape:", X_val_pad.shape)
print("Test shape:", X_test_pad.shape)

print("Words with pretrained embeddings:", found_words)
print(
    "Embedding coverage:",
    round(found_words / min(len(word_index), CNN_VOCAB_SIZE) * 100, 2),
    "%"
)

Training shape: (186636, 256)
Validation shape: (56740, 256)
Test shape: (59981, 256)
Words with pretrained embeddings: 28747
Embedding coverage: 95.82 %


In [17]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout,
    Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping

# Input
input_layer = Input(shape=(CNN_MAX_LEN,))

# Pretrained GloVe embedding layer
embedding_layer = Embedding(
    input_dim=CNN_VOCAB_SIZE,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    trainable=True
)(input_layer)

# Multiple convolution filter widths
pooled_outputs = []

for filter_size in [3, 4, 5]:

    conv = Conv1D(
        filters=128,
        kernel_size=filter_size,
        activation="relu",
        padding="valid"
    )(embedding_layer)

    pooled = GlobalMaxPooling1D()(conv)

    pooled_outputs.append(pooled)

# Concatenate pooled features
merged = Concatenate()(pooled_outputs)

# Classification layers
dense = Dense(128, activation="relu")(merged)

dropout = Dropout(0.5)(dense)

output_layer = Dense(1, activation="sigmoid")(dropout)

cnn_model = Model(
    inputs=input_layer,
    outputs=output_layer
)

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 256, 100)  │  3,000,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 254, 128)  │     38,528 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 253, 128)  │     51,328 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 252, 128)  │     64,128 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     49,280 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        129 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,203,393 (12.22 MB)

 Trainable params: 3,203,393 (12.22 MB)

 Non-trainable params: 0 (0.00 B)

In [18]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

cnn_history = cnn_model.fit(
    X_train_pad,
    y_train,
    validation_data=(X_val_pad, y_val),
    epochs=6,
    batch_size=128,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/6
1459/1459 ━━━━━━━━━━━━━━━━━━━━ 41s 23ms/step - accuracy: 0.7427 - loss: 0.5119 - val_accuracy: 0.6967 - val_loss: 0.7001
Epoch 2/6
1459/1459 ━━━━━━━━━━━━━━━━━━━━ 23s 16ms/step - accuracy: 0.8756 - loss: 0.2974 - val_accuracy: 0.6935 - val_loss: 0.9509
Epoch 3/6
1459/1459 ━━━━━━━━━━━━━━━━━━━━ 24s 16ms/step - accuracy: 0.9237 - loss: 0.1951 - val_accuracy: 0.7362 - val_loss: 0.8824


In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

cnn_val_probs = cnn_model.predict(X_val_pad).ravel()

cnn_val_preds = (
    cnn_val_probs >= 0.5
).astype(int)

print("Validation Accuracy:",
      accuracy_score(y_val, cnn_val_preds))

print("Validation Precision:",
      precision_score(y_val, cnn_val_preds))

print("Validation Recall:",
      recall_score(y_val, cnn_val_preds))

print("Validation F1:",
      f1_score(y_val, cnn_val_preds))

print("Validation ROC-AUC:",
      roc_auc_score(y_val, cnn_val_probs))

print("\nClassification Report:")

print(classification_report(
    y_val,
    cnn_val_preds,
    target_names=["AI-generated", "Human-written"],
    digits=4
))

1774/1774 ━━━━━━━━━━━━━━━━━━━━ 9s 4ms/step
Validation Accuracy: 0.6966866408177652
Validation Precision: 0.6553739261573264
Validation Recall: 0.8464990086611708
Validation F1: 0.7387753862967124
Validation ROC-AUC: 0.7942541369260574

Classification Report:
               precision    recall  f1-score   support

 AI-generated     0.7749    0.5428    0.6384     27991
Human-written     0.6554    0.8465    0.7388     28749

     accuracy                         0.6967     56740
    macro avg     0.7152    0.6947    0.6886     56740
 weighted avg     0.7144    0.6967    0.6893     56740



In [20]:
from sklearn.metrics import confusion_matrix

cnn_test_probs = cnn_model.predict(X_test_pad).ravel()

cnn_test_preds = (
    cnn_test_probs >= 0.5
).astype(int)

print("CNN Test Accuracy:",
      accuracy_score(y_test, cnn_test_preds))

print("CNN Test ROC-AUC:",
      roc_auc_score(y_test, cnn_test_probs))

print("\nClassification Report:")

print(classification_report(
    y_test,
    cnn_test_preds,
    target_names=["AI-generated", "Human-written"],
    digits=4
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, cnn_test_preds))

1875/1875 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step
CNN Test Accuracy: 0.6746136276487554
CNN Test ROC-AUC: 0.7742486732267144

Classification Report:
               precision    recall  f1-score   support

 AI-generated     0.7732    0.5089    0.6138     30478
Human-written     0.6251    0.8458    0.7189     29503

     accuracy                         0.6746     59981
    macro avg     0.6991    0.6774    0.6663     59981
 weighted avg     0.7003    0.6746    0.6655     59981

Confusion Matrix:
[[15510 14968]
 [ 4549 24954]]


In [21]:
import os
import joblib

os.makedirs("saved_models", exist_ok=True)

# Save trained Logistic Regression model
joblib.dump(
    lr_final,
    "saved_models/logistic_regression.pkl"
)

# Save the fitted word TF-IDF vectorizer
joblib.dump(
    tfidf,
    "saved_models/word_tfidf_vectorizer.pkl"
)

print("Logistic Regression and TF-IDF vectorizer saved!")

Logistic Regression and TF-IDF vectorizer saved!


In [22]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

# Save trained SVM
joblib.dump(
    svm_final,
    "saved_models/linear_svm.pkl"
)

# Save word and character vectorizers
joblib.dump(
    tfidf,
    "saved_models/svm_word_tfidf.pkl"
)

joblib.dump(
    char_vectorizer,
    "saved_models/svm_char_tfidf.pkl"
)

print("SVM and both TF-IDF vectorizers saved!")

SVM and both TF-IDF vectorizers saved!


In [23]:
import os
import json

os.makedirs("saved_models/cnn", exist_ok=True)

# Save complete CNN architecture and trained weights
cnn_model.save("saved_models/cnn/kim_cnn.keras")

# Save tokenizer
with open("saved_models/cnn/tokenizer.json", "w") as f:
    f.write(cnn_tokenizer.to_json())

# Save model configuration
cnn_config = {
    "vocab_size": CNN_VOCAB_SIZE,
    "max_length": CNN_MAX_LEN,
    "embedding_dimension": EMBEDDING_DIM,
    "filter_sizes": [3, 4, 5],
    "filters": 128,
    "embedding_source": "GloVe 100d",
    "padding": "post",
    "truncating": "post",
    "classification_threshold": 0.5
}

with open("saved_models/cnn/config.json", "w") as f:
    json.dump(cnn_config, f, indent=4)

print("Kim-style CNN, tokenizer and configuration saved!")

Kim-style CNN, tokenizer and configuration saved!


In [24]:
import pandas as pd

predictions_df = pd.DataFrame({
    "true_label": y_test,
    "logistic_regression_prediction": lr_test_preds,
    "logistic_regression_probability": lr_test_probs,
    "svm_prediction": svm_test_preds,
    "svm_score": svm_test_scores,
    "cnn_prediction": cnn_test_preds,
    "cnn_probability": cnn_test_probs
})

predictions_df.to_csv(
    "saved_models/test_predictions.csv",
    index=False
)

print("Test predictions saved!")

Test predictions saved!


In [25]:
import shutil

shutil.make_archive(
    "all_saved_models",
    "zip",
    root_dir=".",
    base_dir="saved_models"
)

print("All model artifacts packaged successfully!")

All model artifacts packaged successfully!
